[<img src="https://colab.research.google.com/assets/colab-badge.svg" align="left" alt="Open In Colab"/>](https://colab.research.google.com/github/luiscuellarh/apigee-unified-ai-gateway/blob/main/03_apigee_mcp_gateway.ipynb)

<br><br>


# Apigee AI Gateway — MCP Gateway Governance & Policies (AFT)

**Repository Location:** [`https://github.com/luiscuellarh/apigee-unified-ai-gateway/blob/main/03_apigee_mcp_gateway.ipynb`](https://github.com/luiscuellarh/apigee-unified-ai-gateway/blob/main/03_apigee_mcp_gateway.ipynb)  
**Execution Constraint:** This notebook is version-controlled in GitHub and designed to run cleanly when opened directly from that repository path in Google Colab (no local-only paths, no machine-specific assumptions, and no state carried between sessions).

**Framework & Toolkit:**
- **Toolkit:** **AFT (`aft` — Apigee Foundation Toolkit / Apigee Feature Templater)** (`https://github.com/apigee/apigee-templater`)
- **Repository:** [`https://github.com/luiscuellarh/apigee-unified-ai-gateway`](https://github.com/luiscuellarh/apigee-unified-ai-gateway)
- **Assumptions:** Apigee X is already provisioned with `APIGEE_ENV` attached to an environment group (`ENV_GROUP`), and target MCP backend services are reachable.

---

## Step 0 — Preflight (Single-Run Blocking Gate)
Run **Step 0** once to execute all four preflight stages (`0.1`–`0.4`) in a single execution:
- **0.1** Inputs & Global Configuration
- **0.2** Verify Apigee Topology, APIs & IAM
- **0.3** Verify Service Account Roles, Permissions & `<GoogleAccessToken>` Bindings + Live Gemini
- **0.4** Install AFT CLI & Preflight Summary Gate


In [ ]:
#@title 0.1–0.4 Step 0 Preflight (Inputs, Apigee Topology & IAM, Service Account & Live Gemini, AFT CLI & Summary Gate)
# Runs 0.1, 0.2, 0.3, and 0.4 sequentially in a single execution.
import json, os, pathlib, re, shutil, subprocess, time, urllib.error, urllib.request
import requests

# ============================================================================
# 0.1 Inputs & Global Configuration
# ============================================================================
print("▶ [0.1] Inputs & Global Configuration...")
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Authenticated successfully with Google Cloud in Colab.")
except ImportError:
    print("ℹ️ Running in local/standard Jupyter environment.")

#@markdown ### Project & Apigee
PROJECT_ID = "" #@param {type:"string"}
APIGEE_ORG = "" #@param {type:"string"}
APIGEE_ENV = "dev" #@param {type:"string"}
ENV_GROUP = "dev-group" #@param {type:"string"}
REGION = "europe-west1" #@param {type:"string"}
HOSTNAME = "" #@param {type:"string"}

#@markdown ### Service Account
SERVICE_ACCOUNT = "" #@param {type:"string"}

#@markdown ### Gemini
GEMINI_MODEL = "gemini-2.5-flash" #@param {type:"string"}

#@markdown ### Behaviour Flags
ENABLE_APIS = False #@param {type:"boolean"}

PROJECT_ID, APIGEE_ENV, ENV_GROUP, REGION, HOSTNAME, SERVICE_ACCOUNT = (
    v.strip() for v in (PROJECT_ID, APIGEE_ENV, ENV_GROUP, REGION, HOSTNAME, SERVICE_ACCOUNT)
)
APIGEE_ORG = APIGEE_ORG.strip() or PROJECT_ID
APIGEE_ENVIRONMENT = APIGEE_ENV
GEMINI_MODEL = GEMINI_MODEL.strip().removeprefix("google/")

missing = [k for k, v in {"PROJECT_ID": PROJECT_ID, "APIGEE_ENV": APIGEE_ENV, "REGION": REGION, "GEMINI_MODEL": GEMINI_MODEL}.items() if not v]
if missing:
    raise SystemExit(f"❌ HALT: Missing required parameter(s): {", ".join(missing)}.")

# If SERVICE_ACCOUNT is provided, do not ask for a name and do not create a new one
SA_PROVIDED = bool(SERVICE_ACCOUNT)
if SERVICE_ACCOUNT and "@" not in SERVICE_ACCOUNT:
    SERVICE_ACCOUNT = f"{SERVICE_ACCOUNT}@{PROJECT_ID}.iam.gserviceaccount.com"
if SERVICE_ACCOUNT and not re.fullmatch(r"[a-zA-Z0-9._-]+@[a-zA-Z0-9.-]+\.gserviceaccount\.com", SERVICE_ACCOUNT):
    raise SystemExit(f"❌ HALT: Invalid SERVICE_ACCOUNT '{SERVICE_ACCOUNT}' (expected *.gserviceaccount.com).")

os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "PROJECT_ID": PROJECT_ID,
    "APIGEE_ORG": APIGEE_ORG,
    "APIGEE_ENV": APIGEE_ENV,
    "APIGEE_ENVIRONMENT": APIGEE_ENVIRONMENT,
    "ENV_GROUP": ENV_GROUP,
    "REGION": REGION,
    "HOSTNAME": HOSTNAME,
    "SERVICE_ACCOUNT": SERVICE_ACCOUNT,
    "GEMINI_MODEL": GEMINI_MODEL,
})
subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID, "--quiet"], check=True, capture_output=True)

PREFLIGHT = {"inputs": True, "apigee": False, "service_account": False, "gemini": False, "aft": False, "repo": False}
sa_mode = f"{SERVICE_ACCOUNT} (provided — verify only, no creation)" if SA_PROVIDED else f"apigee-ai-gateway-sa@{PROJECT_ID}.iam.gserviceaccount.com (auto-provision if missing)"
print(f"✅ [0.1] Inputs ready | project={PROJECT_ID} | org={APIGEE_ORG} | env={APIGEE_ENV} ({ENV_GROUP}) | region={REGION} | sa={sa_mode}")

# ============================================================================
# 0.2 Verify Apigee Topology, APIs & IAM
# ============================================================================
print("\n▶ [0.2] Verify Apigee Topology, APIs & IAM...")
def get_auth_token() -> str:
    return subprocess.check_output(["gcloud", "auth", "print-access-token"], text=True).strip()

def gcp_request(method: str, url: str, json_body=None, headers=None):
    req_headers = {"Authorization": f"Bearer {get_auth_token()}", "Content-Type": "application/json"}
    if headers:
        req_headers.update(headers)
    try:
        resp = requests.request(method, url, json=json_body, headers=req_headers, timeout=30)
        return resp.status_code, (resp.json() if resp.text else {})
    except ValueError:
        return resp.status_code, {"raw": resp.text}
    except Exception as exc:
        return 0, {"error": str(exc), "url": url}

def wait_for_proxy_ready(proxy_name: str, env_name: str = None, timeout_sec: int = 180) -> str:
    env_target = env_name or APIGEE_ENV
    list_url = f"{APIGEE_BASE}/environments/{env_target}/apis/{proxy_name}/deployments"
    deadline = time.time() + timeout_sec
    last_rev, last_state = None, "NOT_DEPLOYED"
    print(f"⏳ Waiting for proxy '{proxy_name}' to reach READY state in env '{env_target}'...")
    while time.time() < deadline:
        s_list, d_list = gcp_request("GET", list_url)
        deployments = d_list.get("deployments", []) if s_list == 200 else []
        if deployments:
            latest = sorted(deployments, key=lambda d: int(d.get("revision", 0)))[-1]
            last_rev = latest.get("revision")
            rev_url = f"{APIGEE_BASE}/environments/{env_target}/apis/{proxy_name}/revisions/{last_rev}/deployments"
            s_rev, d_rev = gcp_request("GET", rev_url)
            last_state = d_rev.get("state", latest.get("state", "UNKNOWN")) if s_rev == 200 else "UNKNOWN"
            if last_state == "READY":
                print(f"✅ Proxy '{proxy_name}' (revision {last_rev}) is READY in env '{env_target}'.")
                return str(last_rev)
            print(f"   • proxy={proxy_name} | rev={last_rev} | state={last_state} — waiting 5s...")
        else:
            print(f"   • proxy={proxy_name} | state=WAITING_FOR_DEPLOYMENT — waiting 5s...")
        time.sleep(5)
    raise SystemExit(
        f"❌ HALT: Timed out waiting for proxy '{proxy_name}' to become READY in env '{env_target}' "
        f"(last revision={last_rev}, state={last_state})."
    )

ACTIVE_IDENTITY = subprocess.check_output(["gcloud", "config", "get-value", "account"], text=True).strip()
if not ACTIVE_IDENTITY:
    _, t_info = gcp_request("GET", f"https://oauth2.googleapis.com/tokeninfo?access_token={get_auth_token()}")
    ACTIVE_IDENTITY = t_info.get("email", "")
if not ACTIVE_IDENTITY:
    raise SystemExit("❌ HALT: Could not resolve active caller identity.")
caller_member_prefix = "serviceAccount" if ACTIVE_IDENTITY.endswith(".gserviceaccount.com") else "user"
print(f"✅ Active identity: {ACTIVE_IDENTITY}")

# 1. Verify required GCP APIs
REQUIRED_SERVICES = ["apigee.googleapis.com", "cloudresourcemanager.googleapis.com", "iamcredentials.googleapis.com", "aiplatform.googleapis.com"]
disabled_svcs = [
    svc for svc in REQUIRED_SERVICES
    if gcp_request("GET", f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services/{svc}")[1].get("state") != "ENABLED"
]
if disabled_svcs:
    if ENABLE_APIS or disabled_svcs == ["iamcredentials.googleapis.com"]:
        gcp_request("POST", f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services:batchEnable", {"serviceIds": disabled_svcs})
        time.sleep(3)
    else:
        raise SystemExit(f"❌ HALT: Disabled API(s): {", ".join(disabled_svcs)}. Set ENABLE_APIS=True in Step 0 and re-run.")
print(f"✅ Required APIs ENABLED: {", ".join(REQUIRED_SERVICES)}")

# 2. Validate Apigee Organization, Environment, and Environment Group Hostname
APIGEE_BASE = f"https://apigee.googleapis.com/v1/organizations/{APIGEE_ORG}"
status, org_data = gcp_request("GET", APIGEE_BASE)
if status != 200 or org_data.get("state") != "ACTIVE":
    raise SystemExit(f"❌ HALT: Apigee org '{APIGEE_ORG}' not ACTIVE (HTTP {status}): {org_data}")

status, env_data = gcp_request("GET", f"{APIGEE_BASE}/environments/{APIGEE_ENV}")
if status != 200:
    raise SystemExit(f"❌ HALT: Apigee env '{APIGEE_ENV}' not found in org '{APIGEE_ORG}' (HTTP {status}): {env_data}")

_, eg_list = gcp_request("GET", f"{APIGEE_BASE}/envgroups")
matched_group, matched_hosts = None, []
for eg in eg_list.get("environmentGroups", []):
    eg_name = eg["name"]
    if ENV_GROUP and eg_name != ENV_GROUP:
        continue
    _, att_data = gcp_request("GET", f"{APIGEE_BASE}/envgroups/{eg_name}/attachments")
    if APIGEE_ENV in [a.get("environment") for a in att_data.get("environmentGroupAttachments", [])]:
        matched_group, matched_hosts = eg_name, eg.get("hostnames", [])
        break

if not matched_group or (not HOSTNAME and not matched_hosts):
    raise SystemExit(f"❌ HALT: Env '{APIGEE_ENV}' is not attached to a valid envgroup with a hostname in '{APIGEE_ORG}'.")
ENV_GROUP, HOSTNAME = matched_group, (HOSTNAME or matched_hosts[0])
os.environ.update({"ENV_GROUP": ENV_GROUP, "HOSTNAME": HOSTNAME})
print(f"✅ Apigee Topology: org={APIGEE_ORG} | env={APIGEE_ENV} | envgroup={ENV_GROUP} | host={HOSTNAME}")

# 3. Verify Effective Apigee IAM Permissions
REQUIRED_APIGEE_PERMS = [
    "apigee.proxies.create", "apigee.proxies.get", "apigee.proxies.update", "apigee.proxies.delete",
    "apigee.proxyrevisions.get", "apigee.proxyrevisions.update", "apigee.proxyrevisions.delete",
    "apigee.proxyrevisions.deploy", "apigee.proxyrevisions.undeploy",
    "apigee.deployments.create", "apigee.deployments.get", "apigee.deployments.update", "apigee.deployments.delete",
    "apigee.sharedflows.create", "apigee.sharedflows.get", "apigee.sharedflows.delete",
    "apigee.sharedflowrevisions.update", "apigee.sharedflowrevisions.deploy", "apigee.sharedflowrevisions.undeploy",
    "apigee.keyvaluemaps.create", "apigee.keyvaluemaps.get", "apigee.keyvaluemaps.update", "apigee.keyvaluemaps.delete",
    "apigee.keyvaluemapentries.create", "apigee.keyvaluemapentries.get", "apigee.keyvaluemapentries.update", "apigee.keyvaluemapentries.delete",
    "apigee.targetservers.create", "apigee.targetservers.get", "apigee.targetservers.update", "apigee.targetservers.delete",
]
iam_url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}:testIamPermissions"
_, iam_resp = gcp_request("POST", iam_url, {"permissions": REQUIRED_APIGEE_PERMS})
missing_perms = [p for p in REQUIRED_APIGEE_PERMS if p not in set(iam_resp.get("permissions", []))]
if missing_perms:
    raise SystemExit(f"❌ HALT: Missing Apigee IAM permissions: {", ".join(missing_perms)}")
PREFLIGHT["apigee"] = True
print(f"✅ [0.2] Verified all {len(REQUIRED_APIGEE_PERMS)} Apigee IAM permissions.")

# ============================================================================
# 0.3 Verify Service Account Roles, Permissions & <GoogleAccessToken> Bindings + Live Gemini
# ============================================================================
print("\n▶ [0.3] Verify Service Account Roles, Permissions & Bindings + Live Gemini...")
REQUIRED_SA_ROLES = ["roles/aiplatform.user", "roles/iam.serviceAccountTokenCreator"]

def ensure_policy_bindings(get_url: str, set_url: str, required_pairs: list, label: str):
    status, policy = gcp_request("POST", get_url, {})
    if status != 200:
        raise SystemExit(f"❌ HALT: Failed to read {label} IAM policy (HTTP {status}): {policy}")
    bindings = policy.setdefault("bindings", [])
    changed = False
    for member, role in required_pairs:
        entry = next((b for b in bindings if b.get("role") == role and not b.get("condition")), None)
        if entry and member in entry.get("members", []):
            print(f"  ✔ {label} verified: {role} -> {member}")
        else:
            print(f"  ⏳ Binding {label}: {role} -> {member}")
            if entry:
                entry.setdefault("members", []).append(member)
            else:
                bindings.append({"role": role, "members": [member]})
            changed = True
    if changed:
        s_code, s_resp = gcp_request("POST", set_url, {"policy": policy})
        if s_code != 200:
            raise SystemExit(f"❌ HALT: Failed to update {label} IAM policy (HTTP {s_code}): {s_resp}")
        print(f"  ✔ {label} bindings updated.")

# 1. Resolve Project Number & Apigee Service Agent
_, proj_data = gcp_request("GET", f"https://cloudresourcemanager.googleapis.com/v1/projects/{APIGEE_ORG}")
if not proj_data.get("projectNumber"):
    _, proj_data = gcp_request("GET", f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}")
if not proj_data.get("projectNumber"):
    raise SystemExit(f"❌ HALT: Unable to resolve projectNumber for '{PROJECT_ID}': {proj_data}")

PROJECT_NUMBER = str(proj_data["projectNumber"])
APIGEE_SERVICE_AGENT = f"service-{PROJECT_NUMBER}@gcp-sa-apigee.iam.gserviceaccount.com"

# 2. Verify Provided Service Account (or Auto-Provision Default Only When Not Provided)
target_sa = SERVICE_ACCOUNT or f"apigee-ai-gateway-sa@{PROJECT_ID}.iam.gserviceaccount.com"
sa_status, sa_data = gcp_request("GET", f"https://iam.googleapis.com/v1/projects/-/serviceAccounts/{target_sa}")

if SA_PROVIDED:
    if sa_status != 200 or sa_data.get("disabled", False):
        raise SystemExit(f"❌ HALT: Provided SERVICE_ACCOUNT '{SERVICE_ACCOUNT}' is missing or disabled (HTTP {sa_status}).")
    SA_STATUS = "VERIFIED_EXISTING"
    print(f"✅ Verified provided SERVICE_ACCOUNT: {SERVICE_ACCOUNT}")
elif sa_status == 200:
    SERVICE_ACCOUNT, SA_STATUS = target_sa, "REUSED_DEFAULT"
elif sa_status == 404:
    c_status, c_data = gcp_request(
        "POST",
        f"https://iam.googleapis.com/v1/projects/{PROJECT_ID}/serviceAccounts",
        {"accountId": "apigee-ai-gateway-sa", "serviceAccount": {"displayName": "Apigee AI Gateway Service Account"}},
    )
    if c_status not in (200, 201):
        raise SystemExit(f"❌ HALT: Failed to create default service account '{target_sa}' (HTTP {c_status}): {c_data}")
    SERVICE_ACCOUNT, SA_STATUS = c_data.get("email", target_sa), "CREATED_DEFAULT"
    time.sleep(3)
else:
    raise SystemExit(f"❌ HALT: Unable to inspect service account '{target_sa}' (HTTP {sa_status}): {sa_data}")

# 3. Verify & Ensure Project Roles (Gemini + Token Creator) and SA Resource Bindings (<GoogleAccessToken>)
proj_iam_base = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}"
ensure_policy_bindings(
    f"{proj_iam_base}:getIamPolicy",
    f"{proj_iam_base}:setIamPolicy",
    [(f"serviceAccount:{SERVICE_ACCOUNT}", r) for r in REQUIRED_SA_ROLES],
    "Project role",
)

sa_iam_base = f"https://iam.googleapis.com/v1/projects/-/serviceAccounts/{SERVICE_ACCOUNT}"
ensure_policy_bindings(
    f"{sa_iam_base}:getIamPolicy",
    f"{sa_iam_base}:setIamPolicy",
    [
        (f"serviceAccount:{APIGEE_SERVICE_AGENT}", "roles/iam.serviceAccountTokenCreator"),
        (f"serviceAccount:{SERVICE_ACCOUNT}", "roles/iam.serviceAccountTokenCreator"),
        (f"serviceAccount:{SERVICE_ACCOUNT}", "roles/iam.serviceAccountUser"),
        (f"{caller_member_prefix}:{ACTIVE_IDENTITY}", "roles/iam.serviceAccountUser"),
    ],
    "SA <GoogleAccessToken> binding",
)

os.environ["SERVICE_ACCOUNT"] = SERVICE_ACCOUNT
PREFLIGHT["service_account"] = True
print(f"✅ Service Account ready for Gemini & <GoogleAccessToken>: {SERVICE_ACCOUNT} ({SA_STATUS})")

# 4. Verify Live Gemini Connectivity
payload = {"contents": [{"role": "user", "parts": [{"text": "Reply with READY."}]}], "generationConfig": {"maxOutputTokens": 8, "temperature": 0.0}}
def invoke_gemini(loc: str):
    host = "aiplatform.googleapis.com" if loc == "global" else f"{loc}-aiplatform.googleapis.com"
    url = f"https://{host}/v1/projects/{PROJECT_ID}/locations/{loc}/publishers/google/models/{GEMINI_MODEL}:generateContent"
    s, d = gcp_request("POST", url, payload, headers={"x-goog-user-project": PROJECT_ID})
    return gcp_request("POST", url, payload) if s == 403 else (s, d)

VERTEX_LOCATION, gen_status, gen_data = REGION, 0, {}
for _ in range(12):
    gen_status, gen_data = invoke_gemini(REGION)
    if gen_status == 200:
        VERTEX_LOCATION = REGION
        break
    for fb in [loc for loc in ("global", "us-central1") if loc != REGION]:
        fb_s, fb_d = invoke_gemini(fb)
        if fb_s == 200:
            gen_status, gen_data, VERTEX_LOCATION = fb_s, fb_d, fb
            break
    if gen_status == 200 or gen_status not in (403, 404):
        break
    time.sleep(5)

if gen_status != 200:
    raise SystemExit(f"❌ HALT: Live Gemini call failed (HTTP {gen_status}): {gen_data}")

os.environ["VERTEX_LOCATION"] = VERTEX_LOCATION
PREFLIGHT["gemini"] = True
print(f"✅ [0.3] Live Gemini verified: model={GEMINI_MODEL} | location={VERTEX_LOCATION}")

# ============================================================================
# 0.4 Install AFT CLI & Preflight Summary Gate
# ============================================================================
print("\n▶ [0.4] Install AFT CLI & Preflight Summary Gate...")
os.environ.pop("AFT_REPOSITORY", None)
for extra_bin in [str(pathlib.Path.home() / ".aft" / "bin"), str(pathlib.Path.home() / ".local" / "bin")]:
    if extra_bin not in os.environ["PATH"]:
        os.environ["PATH"] = f"{extra_bin}:{os.environ['PATH']}"

if not shutil.which("aft"):
    subprocess.run("curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh", shell=True, check=True)

aft_bin = shutil.which("aft")
if not aft_bin:
    raise SystemExit("❌ HALT: 'aft' binary not found on PATH after installation.")
aft_ver = subprocess.run([aft_bin, "--version"], capture_output=True, text=True, check=True).stdout.strip()
PREFLIGHT["aft"] = True
print(f"✅ AFT CLI ready: {aft_ver} ({aft_bin})")

# 2. Clone GitHub Repository (local templates & features for AFT)
REPO_URL = "https://github.com/luiscuellarh/apigee-unified-ai-gateway.git"
REPO_BRANCH = "main"
REPO_DIR = pathlib.Path.cwd() / "apigee-unified-ai-gateway"

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True, capture_output=True, text=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True, capture_output=True, text=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True, capture_output=True, text=True)

if not (REPO_DIR / "templates").is_dir() or not (REPO_DIR / "features").is_dir():
    raise SystemExit(f"❌ HALT: Cloned repository at '{REPO_DIR}' is missing 'templates/' or 'features/' directories.")

WORKSPACE_DIR = REPO_DIR
os.environ.update({"REPO_URL": REPO_URL, "REPO_BRANCH": REPO_BRANCH, "REPO_DIR": str(REPO_DIR), "WORKSPACE_DIR": str(WORKSPACE_DIR)})
PREFLIGHT["repo"] = True
num_templates = len(list((REPO_DIR / "templates").glob("*.yaml")))
num_features = len(list((REPO_DIR / "features").glob("*.yaml")))
print(f"✅ Cloned GitHub repo ({REPO_BRANCH}): {REPO_DIR} ({num_templates} templates, {num_features} features)")

# 3. Preflight Summary Gate
failed_gates = [k for k, v in PREFLIGHT.items() if not v]
if failed_gates:
    raise SystemExit(f"❌ HALT: Preflight incomplete. Failed gate(s): {", ".join(failed_gates)}")

print("=" * 72)
for label, val in [
    ("GCP Project / No.",  f"{PROJECT_ID} ({PROJECT_NUMBER})"),
    ("Apigee Org / Env",   f"{APIGEE_ORG} / {APIGEE_ENV} ({ENV_GROUP})"),
    ("Gateway Hostname",   HOSTNAME),
    ("Active Identity",    ACTIVE_IDENTITY),
    ("Service Account",    f"{SERVICE_ACCOUNT} ({SA_STATUS}, <GoogleAccessToken> ready)"),
    ("Gemini Readiness",   f"PASS ({GEMINI_MODEL} @ {VERTEX_LOCATION})"),
    ("AFT Toolkit",        f"READY ({aft_ver})"),
    ("Local Repo (AFT)",   f"{REPO_DIR} (branch={REPO_BRANCH}, {num_templates} templates, {num_features} features)"),
]:
    print(f"{label:<20}: {val}")
print("=" * 72)
print("PREFLIGHT PASSED — safe to proceed")


## Step 1 — Deploy MCP Proxy for Loans, API Product & Developer App
- **Purpose:** Deploy the Model Context Protocol (MCP) Gateway proxy for the Loans service (`mcp-gateway-loan-product`), register the **AI Loan Agent Package** (`ai-loan-agent-package`) enforcing MCP operation-level allow-listing and quota (`tools/list` and `tools/call/calculate_monthly_installment` at 30 req/min), and onboard the **Loan Agent App** (`loan-agent@example.com`).
- **AFT Deployments:**
  1. **MCP Loan Proxy:** `aft -i templates/mcp-gateway-loan-product.yaml --organization $APIGEE_ORG --environment $APIGEE_ENVIRONMENT --sa $SERVICE_ACCOUNT`
  2. **API Product / Package:** `aft -i config/products/ai-loan-agent-package.yaml --organization $APIGEE_ORG --environment $APIGEE_ENVIRONMENT --sa $SERVICE_ACCOUNT`
  3. **Developer & App:** `aft -i config/apps/user_apps_mcp_loan.yaml --organization $APIGEE_ORG --environment $APIGEE_ENVIRONMENT --sa $SERVICE_ACCOUNT`
- **Dependencies:** Step 0 (Preflight)

In [ ]:
#@title 1.1 Deploy MCP Proxy for Loans, API Product & Developer App via AFT
# 1) Deploys templates/mcp-gateway-loan-product.yaml via AFT and waits until READY
# 2) Deploys config/products/ai-loan-agent-package.yaml via AFT
# 3) Deploys config/apps/user_apps_mcp_loan.yaml via AFT
# (Also ensures features/auth-oauth-server.yaml is active in APIGEE_ENVIRONMENT for OAuth 2.0 token generation)

os.environ.pop("AFT_REPOSITORY", None)
APIGEE_ENVIRONMENT = APIGEE_ENV
os.environ["APIGEE_ENVIRONMENT"] = APIGEE_ENVIRONMENT

if "wait_for_proxy_ready" not in globals():
    def wait_for_proxy_ready(proxy_name: str, env_name: str = None, timeout_sec: int = 180) -> str:
        env_target = env_name or APIGEE_ENV
        list_url = f"{APIGEE_BASE}/environments/{env_target}/apis/{proxy_name}/deployments"
        deadline = time.time() + timeout_sec
        last_rev, last_state = None, "NOT_DEPLOYED"
        print(f"⏳ Waiting for proxy '{proxy_name}' to reach READY state in env '{env_target}'...")
        while time.time() < deadline:
            s_list, d_list = gcp_request("GET", list_url)
            deployments = d_list.get("deployments", []) if s_list == 200 else []
            if deployments:
                latest = sorted(deployments, key=lambda d: int(d.get("revision", 0)))[-1]
                last_rev = latest.get("revision")
                rev_url = f"{APIGEE_BASE}/environments/{env_target}/apis/{proxy_name}/revisions/{last_rev}/deployments"
                s_rev, d_rev = gcp_request("GET", rev_url)
                last_state = d_rev.get("state", latest.get("state", "UNKNOWN")) if s_rev == 200 else "UNKNOWN"
                if last_state == "READY":
                    print(f"✅ Proxy '{proxy_name}' (revision {last_rev}) is READY in env '{env_target}'.")
                    return str(last_rev)
                print(f"   • proxy={proxy_name} | rev={last_rev} | state={last_state} — waiting 5s...")
            else:
                print(f"   • proxy={proxy_name} | state=WAITING_FOR_DEPLOYMENT — waiting 5s...")
            time.sleep(5)
        raise SystemExit(
            f"❌ HALT: Timed out waiting for proxy '{proxy_name}' to become READY in env '{env_target}' "
            f"(last revision={last_rev}, state={last_state})."
        )

TEMPLATE_LOAN_REL_PATH = "templates/mcp-gateway-loan-product.yaml"
FEATURE_LOAN_ENDPOINT_REL_PATH = "features/mcp-loan-endpoint.yaml"
FEATURE_OAUTH_QUOTA_REL_PATH = "features/api-quota-policy-spike-oauth.yaml"
FEATURE_OAUTH_SERVER_REL_PATH = "features/auth-oauth-server.yaml"
PRODUCT_LOAN_REL_PATH = "config/products/ai-loan-agent-package.yaml"
APPS_LOAN_REL_PATH = "config/apps/user_apps_mcp_loan.yaml"

required_rel_paths = [
    TEMPLATE_LOAN_REL_PATH,
    FEATURE_LOAN_ENDPOINT_REL_PATH,
    FEATURE_OAUTH_QUOTA_REL_PATH,
    FEATURE_OAUTH_SERVER_REL_PATH,
    PRODUCT_LOAN_REL_PATH,
    APPS_LOAN_REL_PATH,
]
if any(not (REPO_DIR / p).is_file() for p in required_rel_paths):
    print(f"🔄 Fetching latest files from origin/{REPO_BRANCH}...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], capture_output=True, text=True)
    for rel_p in required_rel_paths:
        if not (REPO_DIR / rel_p).is_file():
            subprocess.run(["git", "-C", str(REPO_DIR), "checkout", f"origin/{REPO_BRANCH}", "--", rel_p], capture_output=True, text=True)

for rel_p in required_rel_paths:
    if not (REPO_DIR / rel_p).is_file():
        raise SystemExit(f"❌ HALT: Required file not found at '{REPO_DIR / rel_p}'. Ensure it is committed to the repo or re-run Step 0.")

# Ensure OAuth 2.0 token server (auth-oauth-server) is deployed in APIGEE_ENVIRONMENT so /v1/oauth/token can mint tokens
OAUTH_PROXY_NAME = "auth-oauth-server"
oauth_dep_url = f"{APIGEE_BASE}/environments/{APIGEE_ENVIRONMENT}/apis/{OAUTH_PROXY_NAME}/deployments"
oauth_s, oauth_d = gcp_request("GET", oauth_dep_url)
if oauth_s != 200 or not oauth_d.get("deployments"):
    print(f"⏳ Deploying OAuth 2.0 token server ({FEATURE_OAUTH_SERVER_REL_PATH}) to env '{APIGEE_ENVIRONMENT}'...")
    aft_oauth_cmd = [
        aft_bin,
        "-i", FEATURE_OAUTH_SERVER_REL_PATH,
        "--organization", APIGEE_ORG,
        "--environment", APIGEE_ENVIRONMENT,
        "--sa", SERVICE_ACCOUNT,
    ]
    print(f"⏳ Running: {' '.join(aft_oauth_cmd)}")
    oauth_res = subprocess.run(aft_oauth_cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    if oauth_res.returncode != 0:
        err_out = (oauth_res.stderr or oauth_res.stdout).strip()
        raise SystemExit(f"❌ HALT: AFT OAuth server deployment failed (exit {oauth_res.returncode}):\n{err_out}")
    if oauth_res.stdout.strip():
        print(oauth_res.stdout.strip())
    wait_for_proxy_ready(OAUTH_PROXY_NAME, APIGEE_ENVIRONMENT)
else:
    wait_for_proxy_ready(OAUTH_PROXY_NAME, APIGEE_ENVIRONMENT)

# 1. Deploy the MCP Loan Proxy
PROXY_NAME_MCP_LOAN = "mcp-gateway-loan-product"
aft_proxy_cmd = [
    aft_bin,
    "-i", TEMPLATE_LOAN_REL_PATH,
    "--organization", APIGEE_ORG,
    "--environment", APIGEE_ENVIRONMENT,
    "--sa", SERVICE_ACCOUNT,
]
print(f"\n▶ [1/3] Deploying MCP Loan Proxy: {' '.join(aft_proxy_cmd)}")
deploy_proxy_res = subprocess.run(aft_proxy_cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
if deploy_proxy_res.returncode != 0:
    err_out = (deploy_proxy_res.stderr or deploy_proxy_res.stdout).strip()
    raise SystemExit(f"❌ HALT: AFT proxy deployment failed (exit {deploy_proxy_res.returncode}):\n{err_out}")
if deploy_proxy_res.stdout.strip():
    print(deploy_proxy_res.stdout.strip())

deployed_loan_rev = wait_for_proxy_ready(PROXY_NAME_MCP_LOAN, APIGEE_ENVIRONMENT)

# 2. Deploy the API Product / Package (ai-loan-agent-package)
prod_loan_path = REPO_DIR / PRODUCT_LOAN_REL_PATH
prod_loan_text = prod_loan_path.read_text(encoding="utf-8")
if not re.search(rf"(?m)^\s*-\s*{re.escape(APIGEE_ENVIRONMENT)}\s*$", prod_loan_text):
    prod_loan_text = re.sub(
        r"(?m)^(\s*environments:\s*\n)",
        rf"\g<1>  - {APIGEE_ENVIRONMENT}\n",
        prod_loan_text,
        count=1,
    )
    prod_loan_path.write_text(prod_loan_text, encoding="utf-8")

aft_prod_cmd = [
    aft_bin,
    "-i", PRODUCT_LOAN_REL_PATH,
    "--organization", APIGEE_ORG,
    "--environment", APIGEE_ENVIRONMENT,
    "--sa", SERVICE_ACCOUNT,
]
print(f"\n▶ [2/3] Deploying API Product / Package: {' '.join(aft_prod_cmd)}")
deploy_prod_res = subprocess.run(aft_prod_cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
if deploy_prod_res.returncode != 0:
    err_out = (deploy_prod_res.stderr or deploy_prod_res.stdout).strip()
    raise SystemExit(f"❌ HALT: AFT product deployment failed (exit {deploy_prod_res.returncode}):\n{err_out}")
if deploy_prod_res.stdout.strip():
    print(deploy_prod_res.stdout.strip())
print(f"✅ Deployed API Product: {PRODUCT_LOAN_REL_PATH}")

# 3. Deploy the Developer and App (user_apps_mcp_loan.yaml)
aft_apps_cmd = [
    aft_bin,
    "-i", APPS_LOAN_REL_PATH,
    "--organization", APIGEE_ORG,
    "--environment", APIGEE_ENVIRONMENT,
    "--sa", SERVICE_ACCOUNT,
]
print(f"\n▶ [3/3] Deploying Developer & App: {' '.join(aft_apps_cmd)}")
deploy_apps_res = subprocess.run(aft_apps_cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
if deploy_apps_res.returncode != 0:
    err_out = (deploy_apps_res.stderr or deploy_apps_res.stdout).strip()
    raise SystemExit(f"❌ HALT: AFT developer/app deployment failed (exit {deploy_apps_res.returncode}):\n{err_out}")
if deploy_apps_res.stdout.strip():
    print(deploy_apps_res.stdout.strip())
print(f"✅ Deployed Developer & App: {APPS_LOAN_REL_PATH}")

OAUTH_TOKEN_ENDPOINT = f"https://{HOSTNAME}/v1/oauth/token"
MCP_LOAN_ENDPOINT = f"https://{HOSTNAME}/loanservice/mcp"
os.environ.update({
    "PROXY_NAME_MCP_LOAN": PROXY_NAME_MCP_LOAN,
    "OAUTH_TOKEN_ENDPOINT": OAUTH_TOKEN_ENDPOINT,
    "MCP_LOAN_ENDPOINT": MCP_LOAN_ENDPOINT,
})

print("=" * 72)
print(f"✅ Deployed MCP Proxy : {PROXY_NAME_MCP_LOAN} (rev={deployed_loan_rev})")
print(f"✅ API Product        : ai-loan-agent-package (tools/list, tools/call/calculate_monthly_installment)")
print(f"✅ Developer App      : Loan Agent App (loan-agent@example.com)")
print(f"✅ OAuth Token URL    : {OAUTH_TOKEN_ENDPOINT}")
print(f"✅ MCP Loan Endpoint  : {MCP_LOAN_ENDPOINT}")
print("=" * 72)

In [ ]:
#@title 1.2 Test MCP Loan Proxy (Interactive MCP Client UX)
# Interactive MCP JSON-RPC 2.0 Client for testing OAuth 2.0 authentication, MCP tool discovery (tools/list),
# authorized tool execution (calculate_monthly_installment), and unauthorized tool/operation blocking.

#@markdown ### OAuth 2.0 Client Credentials (`Loan Agent App`)
CLIENT_ID = "loan-agent-app-key-123" #@param ["loan-agent-app-key-123", "payment-agent-app-key-123", "invalid-client-id-999", ""] {allow-input: true}
CLIENT_SECRET = "loan-agent-app-secret-123" #@param ["loan-agent-app-secret-123", "payment-agent-app-secret-123", "invalid-secret-999", ""] {allow-input: true}

#@markdown ### MCP JSON-RPC 2.0 Operation
MCP_METHOD = "tools/call" #@param ["tools/list", "tools/call"]
TOOL_NAME = "calculate_monthly_installment" #@param ["calculate_monthly_installment", "get_loan_status", "approve_loan_override"] {allow-input: true}

#@markdown ### Tool Parameters (`calculate_monthly_installment`)
LOAN_AMOUNT = 250000 #@param {type:"number"}
ANNUAL_INTEREST_RATE = 4.5 #@param {type:"number"}
TERM_YEARS = 30 #@param {type:"integer"}
CUSTOM_ARGUMENTS_JSON = "" #@param {type:"string"}

PROXY_NAME_MCP_LOAN = globals().get("PROXY_NAME_MCP_LOAN", "mcp-gateway-loan-product")
OAUTH_TOKEN_ENDPOINT = globals().get("OAUTH_TOKEN_ENDPOINT", f"https://{HOSTNAME}/v1/oauth/token")
MCP_LOAN_ENDPOINT = globals().get("MCP_LOAN_ENDPOINT", f"https://{HOSTNAME}/loanservice/mcp")

# Ensure MCP Loan proxy is READY before testing
wait_for_proxy_ready(PROXY_NAME_MCP_LOAN, APIGEE_ENVIRONMENT)

if "_MCP_LOAN_TOOL_SCHEMAS" not in globals():
    _MCP_LOAN_TOOL_SCHEMAS = {}
if "_MCP_LOAN_SESSION_ID" not in globals():
    _MCP_LOAN_SESSION_ID = None

def _parse_mcp_http_response(resp: requests.Response):
    """Parses either standard application/json or SSE (text/event-stream) JSON-RPC 2.0 responses."""
    ctype = (resp.headers.get("Content-Type") or "").lower()
    raw_text = resp.text.strip()
    if "text/event-stream" in ctype or raw_text.startswith("event:") or raw_text.startswith("data:"):
        events = []
        for line in raw_text.splitlines():
            line_s = line.strip()
            if line_s.startswith("data:"):
                data_part = line_s[len("data:"):].strip()
                if data_part and data_part != "[DONE]":
                    try:
                        events.append(json.loads(data_part))
                    except ValueError:
                        events.append({"raw_data": data_part})
        if events:
            for ev in reversed(events):
                if isinstance(ev, dict) and ("result" in ev or "error" in ev):
                    return ev
            return events[-1]
    try:
        return resp.json()
    except ValueError:
        return {"raw": raw_text}

def _build_tool_arguments(tool_name: str) -> dict:
    if CUSTOM_ARGUMENTS_JSON.strip():
        try:
            return json.loads(CUSTOM_ARGUMENTS_JSON.strip())
        except ValueError as exc:
            raise SystemExit(f"❌ HALT: Invalid JSON in CUSTOM_ARGUMENTS_JSON: {exc}")

    # Default arguments from UX inputs, dynamically adapted if tool schema was discovered via tools/list
    schema_props = _MCP_LOAN_TOOL_SCHEMAS.get(tool_name, {}).get("properties", {})
    if schema_props:
        mapped_args = {}
        for prop_name in schema_props:
            p_low = prop_name.lower()
            if any(k in p_low for k in ("principal", "amount", "loan")):
                mapped_args[prop_name] = LOAN_AMOUNT
            elif any(k in p_low for k in ("rate", "interest")):
                mapped_args[prop_name] = ANNUAL_INTEREST_RATE
            elif "month" in p_low and "year" not in p_low:
                mapped_args[prop_name] = int(TERM_YEARS * 12)
            elif any(k in p_low for k in ("year", "term", "duration", "period")):
                mapped_args[prop_name] = TERM_YEARS
        if mapped_args:
            return mapped_args

    return {
        "principal": LOAN_AMOUNT,
        "annual_interest_rate": ANNUAL_INTEREST_RATE,
        "term_years": TERM_YEARS,
    }

# 1. Mint OAuth 2.0 Bearer Access Token from Apigee (/v1/oauth/token)
client_id_clean = CLIENT_ID.strip()
client_secret_clean = CLIENT_SECRET.strip()
access_token = None

print(f"🔐 OAuth Token Endpoint : {OAUTH_TOKEN_ENDPOINT}")
print(f"🌐 MCP Loan Endpoint    : {MCP_LOAN_ENDPOINT}")
print(f"🆔 Client ID            : {client_id_clean or '(none — unauthenticated request)'}")

if client_id_clean:
    t_oauth0 = time.perf_counter()
    oauth_resp = requests.post(
        OAUTH_TOKEN_ENDPOINT,
        data={"grant_type": "client_credentials"},
        auth=(client_id_clean, client_secret_clean),
        headers={"Accept": "application/json"},
        timeout=30,
    )
    oauth_ms = (time.perf_counter() - t_oauth0) * 1000.0
    if oauth_resp.status_code == 200:
        token_data = oauth_resp.json()
        access_token = token_data.get("access_token")
        expires_in = token_data.get("expires_in", "N/A")
        masked_tok = f"{access_token[:8]}...{access_token[-6:]}" if access_token and len(access_token) > 14 else "issued"
        print(f"✅ OAuth 2.0 Token Minted (HTTP 200 | {oauth_ms:.0f}ms | token={masked_tok} | expires_in={expires_in}s)")
    else:
        print(f"⚠️ OAuth 2.0 Token Request Rejected (HTTP {oauth_resp.status_code} | {oauth_ms:.0f}ms):")
        try:
            print(json.dumps(oauth_resp.json(), indent=2))
        except ValueError:
            print(oauth_resp.text)
        print("   ↳ Proceeding with synthetic invalid Bearer token to verify OA-VerifyAccessToken enforcement on MCP proxy...")
        access_token = "invalid-bearer-token-999"
else:
    print("ℹ️ No CLIENT_ID provided — sending unauthenticated MCP request to verify OA-VerifyAccessToken enforcement.")

mcp_headers = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
}
if access_token:
    mcp_headers["Authorization"] = f"Bearer {access_token}"
if _MCP_LOAN_SESSION_ID:
    mcp_headers["Mcp-Session-Id"] = _MCP_LOAN_SESSION_ID

def _send_mcp_rpc(payload_dict: dict, headers_dict: dict):
    resp = requests.post(
        MCP_LOAN_ENDPOINT,
        json=payload_dict,
        headers=headers_dict,
        timeout=(15, 60),
    )
    resp.encoding = "utf-8"
    sid = resp.headers.get("Mcp-Session-Id") or resp.headers.get("mcp-session-id")
    if sid:
        global _MCP_LOAN_SESSION_ID
        _MCP_LOAN_SESSION_ID = sid
        headers_dict["Mcp-Session-Id"] = sid
    return resp

# If calling calculate_monthly_installment for the first time with a valid token, auto-discover parameter names via tools/list
if (
    MCP_METHOD == "tools/call"
    and TOOL_NAME.strip() == "calculate_monthly_installment"
    and not CUSTOM_ARGUMENTS_JSON.strip()
    and "calculate_monthly_installment" not in _MCP_LOAN_TOOL_SCHEMAS
    and access_token
    and access_token != "invalid-bearer-token-999"
):
    disc_resp = _send_mcp_rpc({"jsonrpc": "2.0", "id": 0, "method": "tools/list", "params": {}}, mcp_headers)
    if disc_resp.status_code == 200:
        disc_body = _parse_mcp_http_response(disc_resp)
        for t_item in (disc_body.get("result") or {}).get("tools", []):
            if isinstance(t_item, dict) and t_item.get("name"):
                _MCP_LOAN_TOOL_SCHEMAS[t_item["name"]] = t_item.get("inputSchema") or {}

# 2. Construct MCP JSON-RPC 2.0 Request Payload
if MCP_METHOD == "tools/list":
    mcp_payload = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/list",
        "params": {},
    }
else:
    tool_args = _build_tool_arguments(TOOL_NAME.strip())
    mcp_payload = {
        "jsonrpc": "2.0",
        "id": 2,
        "method": "tools/call",
        "params": {
            "name": TOOL_NAME.strip(),
            "arguments": tool_args,
        },
    }

print("\n📤 MCP JSON-RPC 2.0 Request Payload:")
print(json.dumps(mcp_payload, indent=2))
print("-" * 72)

t0 = time.perf_counter()
try:
    resp = _send_mcp_rpc(mcp_payload, mcp_headers)
    latency_ms = (time.perf_counter() - t0) * 1000.0
    parsed_body = _parse_mcp_http_response(resp)

    if resp.status_code != 200:
        status_label = {
            401: "🔒 UNAUTHORIZED (Rejected by OA-VerifyAccessToken or Product Authorization)",
            403: "🚫 FORBIDDEN (MCP Operation / Tool Not Allowed by API Product 'ai-loan-agent-package')",
            429: "🚨 QUOTA / SPIKE ARREST EXCEEDED (QU-EnforceQuota or SA-SpikeArrest-30qps Triggered)",
        }.get(resp.status_code, "🛡️ BLOCKED / REJECTED BY MCP GATEWAY")
        print(f"{status_label} (HTTP {resp.status_code} | latency={latency_ms:.0f}ms):")
        print(json.dumps(parsed_body, indent=2))
    else:
        print(f"✅ ALLOWED BY MCP GATEWAY (HTTP {resp.status_code} | method={MCP_METHOD} | latency={latency_ms:.0f}ms)")
        if "error" in parsed_body:
            print("\n⚠️ MCP Server returned a JSON-RPC error:")
            print(json.dumps(parsed_body["error"], indent=2))
        elif MCP_METHOD == "tools/list":
            tools_list = (parsed_body.get("result") or {}).get("tools", [])
            allowed_tools = {"calculate_monthly_installment"}
            print(f"\n🛠️ Discovered {len(tools_list)} MCP Tool(s) on Loan Server:")
            for idx, t_obj in enumerate(tools_list, 1):
                t_name = t_obj.get("name", "unknown")
                t_desc = (t_obj.get("description") or "").strip().splitlines()[0] if t_obj.get("description") else "No description"
                t_schema = t_obj.get("inputSchema") or {}
                _MCP_LOAN_TOOL_SCHEMAS[t_name] = t_schema
                props = list((t_schema.get("properties") or {}).keys())
                policy_badge = "✅ ALLOWED in ai-loan-agent-package (30/min)" if t_name in allowed_tools else "🚫 RESTRICTED (not in ai-loan-agent-package)"
                print(f"  {idx}. {t_name} [{policy_badge}]")
                print(f"     • Description : {t_desc}")
                print(f"     • Parameters  : {props}")
            print("\n📥 Full JSON-RPC Result:")
            print(json.dumps(parsed_body, indent=2))
        else:
            result_obj = parsed_body.get("result") or {}
            content_items = result_obj.get("content", [])
            print(f"\n🧮 Tool Execution Result ({TOOL_NAME.strip()}):")
            for item in content_items:
                if isinstance(item, dict) and item.get("type") == "text":
                    txt = item.get("text", "")
                    try:
                        print(json.dumps(json.loads(txt), indent=2))
                    except ValueError:
                        print(txt)
                else:
                    print(json.dumps(item, indent=2))
            if not content_items:
                print(json.dumps(parsed_body, indent=2))
except requests.RequestException as exc:
    raise SystemExit(
        f"❌ HALT: Network/TLS error calling '{MCP_LOAN_ENDPOINT}': {exc}\n"
        f"Verify that load balancer IP/DNS and managed SSL certificate for '{HOSTNAME}' are ACTIVE."
    )